# GTFS → GeoJSON Extractor

Extracts two GeoJSON files from a GTFS `.zip`:
- **`network.geojson`** — route shapes (LineStrings), one feature per `shape_id`
- **`stops.geojson`** — all stops (Points) with their attributes

**Requirements:** `pandas`, `shapely`
```
pip install pandas shapely
```

In [1]:
# ── Configuration ──────────────────────────────────────────────────────────────
GTFS_ZIP      = "../../data/mobility/AC-GTFS.zip"          # Path to your GTFS zip file
NETWORK_OUT   = "../../data/mobility/ac-network.geojson"   # Output path for route shapes
STOPS_OUT     = "../../data/mobility/ac-stops.geojson"     # Output path for stops
# ───────────────────────────────────────────────────────────────────────────────

In [2]:
import json
import zipfile
import io
import pandas as pd
from shapely.geometry import LineString, mapping

print("Dependencies loaded ✓")

Dependencies loaded ✓


## 1. Load GTFS files from zip

In [3]:
def read_gtfs_file(zf: zipfile.ZipFile, filename: str) -> pd.DataFrame | None:
    """Read a CSV from inside a GTFS zip, searching any subdirectory."""
    names = zf.namelist()
    # Try exact match first, then suffix match
    matches = [n for n in names if n == filename or n.endswith("/" + filename)]
    if not matches:
        return None
    with zf.open(matches[0]) as f:
        return pd.read_csv(io.TextIOWrapper(f, encoding="utf-8-sig"))


with zipfile.ZipFile(GTFS_ZIP) as zf:
    shapes = read_gtfs_file(zf, "shapes.txt")
    stops  = read_gtfs_file(zf, "stops.txt")
    routes = read_gtfs_file(zf, "routes.txt")   # optional – for route metadata
    trips  = read_gtfs_file(zf, "trips.txt")    # optional – links shapes → routes

print(f"stops.txt  : {len(stops):,} rows")
print(f"shapes.txt : {len(shapes):,} rows" if shapes is not None else "shapes.txt : NOT FOUND")
print(f"routes.txt : {len(routes):,} rows" if routes is not None else "routes.txt : NOT FOUND")
print(f"trips.txt  : {len(trips):,} rows"  if trips  is not None else "trips.txt  : NOT FOUND")

stops.txt  : 4,160 rows
shapes.txt : 237,885 rows
routes.txt : 80 rows
trips.txt  : 13,099 rows


## 2. Build network GeoJSON from `shapes.txt`

Each unique `shape_id` becomes one `LineString` feature.
If `routes.txt` and `trips.txt` are available, route metadata (name, colour, type) is joined in.

In [4]:
ROUTE_TYPE_LABELS = {
    0: "Tram", 1: "Subway", 2: "Rail", 3: "Bus",
    4: "Ferry", 5: "Cable Car", 6: "Gondola", 7: "Funicular",
    11: "Trolleybus", 12: "Monorail",
}

def build_network_geojson(shapes: pd.DataFrame,
                          routes: pd.DataFrame | None,
                          trips:  pd.DataFrame | None) -> dict:
    """
    Convert shapes.txt (and optionally routes/trips) to a GeoJSON FeatureCollection.
    One feature per shape_id, geometry = LineString.
    """
    # Sort points correctly
    shapes = shapes.sort_values(["shape_id", "shape_pt_sequence"])

    # Build a lookup: shape_id → route metadata
    shape_meta: dict[str, dict] = {}
    if trips is not None and routes is not None:
        # One representative route per shape_id
        trip_shape = trips[["shape_id", "route_id"]].drop_duplicates("shape_id").copy()
        trip_shape["route_id"] = trip_shape["route_id"].astype(str)
        routes["route_id"] = routes["route_id"].astype(str)
        merged = trip_shape.merge(routes, on="route_id", how="left")
        for _, row in merged.iterrows():
            props = {}
            for col in ["route_id", "route_short_name", "route_long_name",
                        "route_type", "route_color", "route_text_color",
                        "route_desc", "agency_id"]:
                if col in row.index and pd.notna(row[col]):
                    props[col] = row[col]
            if "route_type" in props:
                props["route_type_label"] = ROUTE_TYPE_LABELS.get(
                    int(props["route_type"]), "Unknown"
                )
            shape_meta[str(row["shape_id"])] = props

    features = []
    for shape_id, group in shapes.groupby("shape_id", sort=False):
        coords = list(zip(group["shape_pt_lon"], group["shape_pt_lat"]))
        if len(coords) < 2:
            continue  # degenerate shape – skip
        geom = mapping(LineString(coords))
        props = {"shape_id": str(shape_id)}
        props.update(shape_meta.get(str(shape_id), {}))
        features.append({"type": "Feature", "geometry": geom, "properties": props})

    return {"type": "FeatureCollection", "features": features}


if shapes is not None:
    network_geojson = build_network_geojson(shapes, routes, trips)
    print(f"Network features : {len(network_geojson['features']):,} shapes")
else:
    print("⚠️  shapes.txt not found – network GeoJSON will not be created.")
    print("   Some GTFS feeds omit shapes.txt; you can fall back to trip/stop sequences.")
    network_geojson = None

Network features : 205 shapes


## 3. Build stops GeoJSON from `stops.txt`

In [5]:
LOCATION_TYPE_LABELS = {
    0: "Stop / Platform",
    1: "Station",
    2: "Station Entrance/Exit",
    3: "Generic Node",
    4: "Boarding Area",
}

def build_stops_geojson(stops: pd.DataFrame) -> dict:
    """Convert stops.txt to a GeoJSON FeatureCollection of Points."""
    # Drop rows without valid coordinates
    valid = stops.dropna(subset=["stop_lat", "stop_lon"]).copy()
    skipped = len(stops) - len(valid)
    if skipped:
        print(f"  Skipped {skipped} stop(s) with missing coordinates")

    optional_cols = [
        "stop_code", "stop_desc", "zone_id", "stop_url",
        "location_type", "parent_station", "stop_timezone",
        "wheelchair_boarding", "level_id", "platform_code",
    ]

    features = []
    for _, row in valid.iterrows():
        props = {
            "stop_id"  : str(row["stop_id"]),
            "stop_name": str(row.get("stop_name", "")),
        }
        for col in optional_cols:
            if col in row.index and pd.notna(row[col]):
                props[col] = row[col]
        if "location_type" in props:
            props["location_type_label"] = LOCATION_TYPE_LABELS.get(
                int(props["location_type"]), "Unknown"
            )
        feature = {
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [float(row["stop_lon"]), float(row["stop_lat"])],
            },
            "properties": props,
        }
        features.append(feature)

    return {"type": "FeatureCollection", "features": features}


stops_geojson = build_stops_geojson(stops)
print(f"Stop features    : {len(stops_geojson['features']):,} stops")

Stop features    : 4,160 stops


## 4. Write output files

In [6]:
def write_geojson(data: dict, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    size_kb = __import__("os").path.getsize(path) / 1024
    print(f"Written → {path}  ({size_kb:,.1f} KB)")


if network_geojson is not None:
    write_geojson(network_geojson, NETWORK_OUT)

write_geojson(stops_geojson, STOPS_OUT)

print("\nDone! ✓")

Written → ../../data/mobility/ac-network.geojson  (16,702.4 KB)
Written → ../../data/mobility/ac-stops.geojson  (1,710.8 KB)

Done! ✓


## 5. Quick sanity-check (optional)

Print a sample feature from each output to verify the structure.

In [7]:
if network_geojson and network_geojson["features"]:
    sample = network_geojson["features"][0].copy()
    coords = sample["geometry"]["coordinates"]
    sample["geometry"]["coordinates"] = f"[{len(coords)} coordinate pairs]"
    print("Network sample feature:")
    print(json.dumps(sample, indent=2))

print()
if stops_geojson["features"]:
    print("Stops sample feature:")
    print(json.dumps(stops_geojson["features"][0], indent=2))

Network sample feature:
{
  "type": "Feature",
  "geometry": {
    "type": "LineString",
    "coordinates": "[1051 coordinate pairs]"
  },
  "properties": {
    "shape_id": "shp-12-05",
    "route_id": "12",
    "route_short_name": "12",
    "route_long_name": "MLK Jr. - Temescal - Grand",
    "route_type": 3,
    "route_color": "2B589C",
    "route_text_color": "FFFFFF",
    "agency_id": 1,
    "route_type_label": "Bus"
  }
}

Stops sample feature:
{
  "type": "Feature",
  "geometry": {
    "type": "Point",
    "coordinates": [
      -122.272992,
      37.768814
    ]
  },
  "properties": {
    "stop_id": "2",
    "stop_name": "8th St & Portola Av",
    "stop_code": 52246,
    "zone_id": "E",
    "stop_url": "https://www.actransit.org/stop/52246",
    "wheelchair_boarding": 0
  }
}
